In [1]:
import math
import numpy as np
import gymnasium as gym
import matplotlib.pyplot as plt

from itertools import count
from importnb import Notebook

with Notebook():
    from LabLatencyModel import LatencyModel, MultiDULatencyModel
    from LabCacheEngine import CacheEngineEnv
    from LabUserRequest import UserRequestEvents
    from LabPrefetchScheduler import PrefetchScheduler

In [ ]:
class EnvWrapper(gym.Env):
    """
    Simple wrapper that delegates all calls to an inner env.
    Subclass this to create your own wrappers.
    """

    metadata = {"render.modes": []}

    def __init__(
        self, 
        n: int,
        m: int,
        n_layers: int,
        lam: float,
        theta: float,
        users_env: None,
        du_caches: None,
        mec_cache: None,
        latency_model: None,
        prefetch_fn: None,
        reward_fn: None,
        max_steps: int = 10000,
        *,
        step_duration_s=1.0
    ):
        super().__init__()

        self.step_count = 0
        self.step_duration_s = step_duration_s
        self.max_steps = max_steps

        self.n = n  # number of tiles per row/column
        self.m = m  # number of tiles per row/column
        self.n_layers = n_layers  # number of layers (base + enhancement)
        
        self.gain_if_prefetched = 1.0
        self.loss_if_not_prefetched = -1.0

        self.theta = theta
        self.lam = lam

        self.users_env = users_env
        self.du_caches = du_caches
        self.mec_cache = mec_cache
        self.latency_model = latency_model

        self.prefetch_fn = prefetch_fn or (lambda cache, action: cache.drl_prefetching(action))
        self.reward_fn = reward_fn or (lambda info: info.get('reward_per_user', {}).get(info.get('current_user', -1), 0.0))

        # History holders
        self.users_reward = {
            u: [] for u in range(self.users_env.n_users)
        }
        self.users_psnr = {
            u: [] for u in range(self.users_env.n_users)
        }
        self.total_gop_requests_per_user = {
            u: 0 for u in range(self.users_env.n_users)
        }
        
        self.scheduler = PrefetchScheduler(
            R_M_D=self.latency_model.R_M_D,
            R_C_M=self.latency_model.R_C_M,
            U=self.latency_model.max_U,
            step_duration_s=self.step_duration_s
        )
    
    # ─────────────────────────────────────────────────────────────────────────
    # Internal Helpers
    # ─────────────────────────────────────────────────────────────────────────
    def _make_ready_bitmaps(self, du_planned, mec_planned):
        du_ready = None
        mec_ready = None

        if du_planned:
            du_ready = []
            for du_idx, bm in enumerate(du_planned):
                cache_key = f"DU:{du_idx}"
                du_ready.append(self.scheduler.materialize_ready_bitmap(cache_key, bm))

        if mec_planned is not None:
            mec_ready = self.scheduler.materialize_ready_bitmap("MEC", mec_planned)

        return du_ready, mec_ready

    def sample_action(self):
        tiles = np.zeros(self.n * self.m, dtype=int)
        c = self.n // 2
        if self.n % 2 == 1:
            center_idx = c * self.n + c
            tiles[center_idx] = 1
        else:
            centers = [(c-1, c-1), (c-1, c), (c, c-1), (c, c)]
            for x, y in centers:
                tiles[y * self.n + x] = 1

        return {
            'video': np.random.randint(0, self.users_env.n_videos),
            'gop': np.random.randint(0, self.users_env.n_gops),
            'tiles': tiles.tolist()
        }, c * self.n + c

    def step(self, actions):
        info = {}
        reqs = []
        user_visited = set()
        
        # -------------------------------------------------------
        # 1: Apply prefetching for all actions this step
        # -------------------------------------------------------
        du_bitmaps_planned = None
        mec_bitmap_planned = None

        while actions and len(actions) > 0:
            action = actions.pop()
            user_visited.add(action["user"])

            # Apply to DU caches
            if self.du_caches and len(self.du_caches) > 0:
                du_bitmaps_planned = [ 
                    self.prefetch_fn(cache, action) for cache in self.du_caches
                ]
            
            # Apply to MEC cache
            if self.mec_cache:
                mec_bitmap_planned = self.prefetch_fn(self.mec_cache, action)

            # Immediate user request
            req = self.users_env.step_single_user(
                action["user"], 
                du_bitmaps_planned, 
                mec_bitmap_planned
            )

            if req is not None:
                reqs.append(req)

        # -------------------------------------------------------
        # 2. Schedule PLANNED tiles (prefetch latency modeling)
        # -------------------------------------------------------
        self.scheduler.schedule_from_mec_plan(mec_bitmap_planned)
        self.scheduler.schedule_from_du_plans(du_bitmaps_planned, mec_bitmap_planned)

        # -------------------------------------------------------
        # 3: Convert PLANNED → READY bitmaps
        # -------------------------------------------------------
        du_ready_bitmaps, mec_ready_bitmap = self._make_ready_bitmaps(
            du_bitmaps_planned, mec_bitmap_planned
        )

        # -------------------------------------------------------
        # 4. Remaining users requests
        # -------------------------------------------------------
        reqs += self.users_env.step(du_ready_bitmaps, mec_ready_bitmap, user_visited)
        info["users_requests"] = reqs

        ## -------------------------------------------------------
        # 5. Compute latency per user (DU/MEC/CLOUD)
        # -------------------------------------------------------
        latency_per_user, bw_cost_per_user = self.compute_latency_and_bw(reqs)
        info["latency_per_user"] = latency_per_user
        info["bandwidth_cost_per_user"] = bw_cost_per_user
        info["avg_latency"] = float(np.mean(list(latency_per_user.values()))) if latency_per_user else 0.0

        # -------------------------------------------------------
        # 6. Cache stats (HIT / MISS)
        # -------------------------------------------------------
        info.update(self.compute_cache_stats(reqs))

        # -------------------------------------------------------
        # 7. PSNR reward
        # -------------------------------------------------------
        psnr_per_user = self.compute_psnr(reqs)
        info["average_psnr"] = psnr_per_user

        # -------------------------------------------------------
        # 8. Compute final per-user reward
        # -------------------------------------------------------
        reward_per_user = self.reward_fn(self, psnr_per_user)

        for u, r in reward_per_user.items():
            self.users_reward[u].append(r)

        info["reward_per_user"] = {
            u: np.mean(vals) if len(vals) else 0.0
            for u, vals in self.users_reward.items()
        }

        # Final total reward over all users that had requests
        active_users = set(req["u"] for req in reqs)
        reward = np.mean([reward_per_user[u] for u in active_users]) if active_users else 0.0

        # -------------------------------------------------------
        # 9. Cache Utilization
        # -------------------------------------------------------
        used = 0
        total = 0
        for cache in self.du_caches:
            used += cache.get_current_capacity()
            total += cache.max_capacity
        if self.mec_cache:
            used += self.mec_cache.get_current_capacity()
            total += self.mec_cache.max_capacity

        info["cache_current_capacity"] = used
        info["cache_total_capacity"] = total
        info["cache_utilization"] = used / total if total else 0.0

        # -------------------------------------------------------
        # 10. Advance simulation time (1 step)
        # -------------------------------------------------------
        self.scheduler.tick()

        done = self.users_env.all_users_done() or (self.step_count >= self.max_steps - 1)
        self.step_count += 1

        return {}, reward, done, info

    # ---------------------------------------------------------
    #  LATENCY + BANDWIDTH COST
    # ---------------------------------------------------------
    def compute_latency_and_bw(self, reqs):

        latency_per_user = {u: 0.0 for u in range(self.users_env.n_users)}
        bw_cost_per_user = {u: 0.0 for u in range(self.users_env.n_users)}

        for req in reqs:
            u = req["u"]

            for tile in req["tiles"]:
                layer = tile["layer"]
                size = self.mec_cache.tile_size_bytes[layer]

                du_hit = tile["events"]["alpha_p_u"]
                mec_hit = tile["events"]["alpha_M_u"]

                # -------------------------
                # Classify event
                # -------------------------
                if du_hit == 1:
                    event = "DU"
                    per_byte_latency = self.latency_model.TD_U
                elif mec_hit == 1:
                    event = "MEC"
                    per_byte_latency = self.latency_model.R_M_D
                else:
                    event = "CLOUD"
                    per_byte_latency = self.latency_model.R_C_M

                # -----------------------------------------------
                # Latency: Eq. (6–7)  (tile_size × per_byte_latency)
                # -----------------------------------------------
                tile_latency = per_byte_latency * size
                latency_per_user[u] += tile_latency

                # -----------------------------------------------
                # Bandwidth cost = size / latency_per_byte = rate * bytes
                # But easier: BW cost = size  (if counting bytes transferred)
                # Or use bitrate = bytes / latency
                # -----------------------------------------------
                bytes_transferred = size
                bw_cost_per_user[u] += bytes_transferred

        return latency_per_user, bw_cost_per_user

    # ---------------------------------------------------------
    #  PSNR MODEL
    # ---------------------------------------------------------
    def compute_psnr(self, reqs):

        psnr_per_user = {}
        for req in reqs:
            u = req["u"]

            base_sum = 0
            enh_sum = 0

            for tile in req["tiles"]:
                l = tile["layer"]
                du_hit = tile["events"]["alpha_p_u"]
                mec_hit = tile["events"]["alpha_M_u"]

                hit = du_hit == 1 or mec_hit == 1

                if l == 0 and hit:
                    base_sum += 30.0
                elif l == 1 and hit:
                    enh_sum += 10.0

            psnr = (base_sum / 12.0) + (enh_sum / 4.0)
            self.users_psnr[u].append(psnr)
            psnr_per_user[u] = psnr

        return psnr_per_user
    
    # ---------------------------------------------------------
    # REWARD FUNCTION
    # ---------------------------------------------------------
    def compute_reward(self, psnr_per_user):
        reward_per_user = {}
        for u in range(self.users_env.n_users):
            psnr = psnr_per_user.get(u, 0.0)

            # Example reward: weighted sum of PSNR and negative latency
            reward_per_user[u] = self.gain_if_prefetched * psnr

        return reward_per_user

    # ---------------------------------------------------------
    #  HIT / MISS STATS
    # ---------------------------------------------------------
    def compute_cache_stats(self, reqs):

        base_hits = 0
        enh_hits  = 0
        base_miss = 0
        enh_miss  = 0

        for req in reqs:
            for tile in req["tiles"]:
                l = tile["layer"]
                hit = tile["events"]["alpha_p_u"] or tile["events"]["alpha_M_u"]

                if l == 0:
                    base_hits += int(hit)
                    base_miss += int(not hit)
                else:
                    enh_hits += int(hit)
                    enh_miss += int(not hit)

        return dict(
            base_layer_hits=base_hits,
            enh_layer_hits=enh_hits,
            base_layer_misses=base_miss,
            enh_layer_misses=enh_miss
        )

    # ---------------------------------------------------------
    # RESET
    # ---------------------------------------------------------
    def reset(self, **kwargs):
        self.step_count = 0
        
        self.users_reward = {
            u: [] for u in range(self.users_env.n_users)
        }
        self.users_psnr = {
            u: [] for u in range(self.users_env.n_users)
        }
        self.total_gop_requests_per_user = {
            u: 0 for u in range(self.users_env.n_users)
        }

        _, info_users = self.users_env.reset(**kwargs)

        info_cache_du = [cache.reset(**kwargs)[1] for cache in self.du_caches]
        info_cache_mec = self.mec_cache.reset(**kwargs)[1] if self.mec_cache else {}

        info_cache = {
            "du_caches_info": info_cache_du,
            **info_cache_mec,
            **info_users
        }

        # Reset scheduler's time and availability
        self.scheduler.now_s = 0.0
        self.scheduler.availability = {}

        return None, info_cache

In [3]:
def getTiles(step, user, users_viewport_tiles, n) -> np.ndarray:
    mask = np.zeros(n * n, dtype=int)    
    for tx, ty in users_viewport_tiles[user][step]:
        if 0 <= tx < n and 0 <= ty < n:
            mask[ty * n + tx] = 1
    
    return mask

In [4]:
if __name__ == "__main__":
    n_episodes = 100
    n_nodes = 3
    n_users = 100
    step_size = 5.0
    alpha = 1.0
    n_gops = 60
    n_layers = 2
    n = 4
    max_capacity = 5000e6  # 5000 MB
    n_videos = 1000

    #### CPT parameters ####
    lam = 3.7183
    theta = 0.5

    users_env = UserRequestEvents(
        n_nodes=n_nodes,
        n_users=n_users,
        step_size=step_size,
        n_videos=n_videos,
        n_gops=n_gops,
        n_layers=n_layers,
        n_tiles=n*n,
        n=n,
        alpha=alpha,
        users_viewport_tiles=None,
        requested_videos=None
    )

    # du_caches = [
    #     CacheEngineEnv(
    #         n_tiles=n*n,
    #         n_videos=n_videos,
    #         cache_capacity=max_capacity
    #     ) for _ in range(n_nodes)  # Number of DUs = n_nodes
    # ]
    du_caches = []

    mec_cache = CacheEngineEnv(
        n_tiles=n*n,
        n_videos=n_videos,
        cache_capacity=max_capacity
    )

    # Create latency model (replace numbers with your real config)
    P = n_nodes; max_U = n_users
    lat_model = MultiDULatencyModel(
        P=P, 
        max_U=max_U,
        R_M_D=80e6,       # 640 Mbps -> 80e6 B/s 
        R_C_M=1.25e9,     # 10 Gbps -> 1.25e9 B/s
        mu=2e7, 
        eta=2e5,
        B_pu_matrix=np.full((P, max_U), 40e6, dtype=float),    # 320 Mbps -> 40e6 B/s
        gamma_pu_matrix=np.full((P, max_U), 5.0, dtype=float), # SNRs
        rhoT_p=[0.2], 
        lambda_p=[0.05],
        du_fixed_delay=0.001,    # 1 ms
        mec_fixed_delay=0.005,   # 5 ms
        cloud_fixed_delay=0.05   # 50 ms
    )

    env = EnvWrapper(
        n=n,
        n_layers=n_layers,
        users_env=users_env, 
        du_caches=du_caches,
        mec_cache=mec_cache,
        latency_model=lat_model,
        lam=lam,
        theta=theta
    )

    print(
        f"Experiment ===================\n"
        f"Total users: {n_users}\n"
        f"Cache capacity: {max_capacity}MB\n"
        f"Video matrix: {n_videos}x{n_layers}x{n*n}\n"
        f"==============================\n"
    )

    _, info = env.reset()
    viewport_tiles = info['viewport_tiles']
    requested_videos = info['requested_videos']

    done = False
    results = []
    total_reward = 0.0

    for step in count():
        actions = [
            {
                'user': user,
                'video': requested_videos[user],
                'tiles': getTiles(step % n_gops, user, viewport_tiles, n),
                'gop': step % n_gops
            } for user in range(n_users)
        ]

        obs, reward, done, info = env.step(actions)

        total_reward += float(reward)

        results.append({
            "step": step,
            "total_reward": total_reward,
            "cache_hits": info["base_layer_cache_hits"] + info["enh_layer_cache_hits"],
            "cache_misses": info["base_layer_cache_misses"] + info["enh_layer_cache_misses"],
            "info": info
        })

        print(
            f"Step {step} - "
            f"Reward: {reward:.4f} - "
            f"Total Reward: {total_reward:.4f} - "
            f"Cache Hits: {info['base_layer_cache_hits'] + info['enh_layer_cache_hits']} - "
            f"Cache Misses: {info['base_layer_cache_misses'] + info['enh_layer_cache_misses']}\n"
            f"Cache Utilization: {info['cache_utilization']:.2%} - "
            f"Items in Cache: {info['cache_num_items']} - "
            f"Final Capacity: {info['cache_current_capacity']/1e6:.2f}/{info['cache_total_capacity']/1e6:.1f} MB"
        )
        
        if done: 
            break

TypeError: UserRequestEvents.__init__() got an unexpected keyword argument 'alpha'

In [ ]:
if __name__ == "__main__":
    steps = range(1, len(results) + 1)
    cache_hits_series = [r["cache_hits"] for r in results]
    cache_misses_series = [r["cache_misses"] for r in results]

    fig, axes = plt.subplots(1, 3, figsize=(18, 4), sharex=True)

    # Rewards with moving average
    axes[0].plot(steps, [r["total_reward"] for r in results], label="Step reward", alpha=0.7, color="blue")
    w = max(1, min(20, len(results) // 10))
    if w > 1:
        ma = [sum([r["total_reward"] for r in results][i - w:i]) / w for i in range(w, len(results) + 1)]
        axes[0].plot(range(w, len(results) + 1), ma, label=f"Moving avg (w={w})", color="orange")
    axes[0].set_xlabel("Step")
    axes[0].set_ylabel("Total reward")
    axes[0].set_title("Training Rewards")
    axes[0].grid(True, alpha=0.3)
    axes[0].legend()

    # Cache hits
    axes[1].plot(steps, cache_hits_series, label="Cache hits", color="green", alpha=0.8)
    axes[1].set_title("Cache Hits per Step")
    axes[1].set_xlabel("Step")
    axes[1].set_ylabel("Hits")
    axes[1].grid(True, alpha=0.3)
    axes[1].legend()

    # Cache misses
    axes[2].plot(steps, cache_misses_series, label="Cache misses", color="red", alpha=0.8)
    axes[2].set_title("Cache Misses per Step")
    axes[2].set_xlabel("Step")
    axes[2].set_ylabel("Misses")
    axes[2].grid(True, alpha=0.3)
    axes[2].legend()

    plt.tight_layout()
    plt.show()

In [ ]:
if __name__ == "__main__":
    n_episodes = 100
    n_nodes = 3
    n_users = 100
    step_size = 5.0
    alpha = 1.0
    n_gops = 60
    n_layers = 2
    n = 4
    max_capacity = 5000e6  # 5000 MB
    n_videos = 1000

    #### CPT parameters ####
    lam = 3.7183
    theta = 0.5

    users_env = UserTileRequestEvents(
        n_users=n_users,
        step_size=step_size,
        n_videos=n_videos,
        n_gops=n_gops,
        n_layers=n_layers,
        n_tiles=n*n,
        n=n,
        alpha=alpha,
        users_viewport_tiles=None,
        requested_videos=None
    )

    du_caches = []

    mec_cache = CacheEngineEnv(
        n_tiles=n*n,
        n_videos=n_videos,
        cache_capacity=max_capacity
    )

    # Create latency model (replace numbers with your real config)
    P = n_nodes; max_U = n_users
    lat_model = MultiDULatencyModel(
        P=P, 
        max_U=max_U,
        R_M_D=80e6,       # 640 Mbps -> 80e6 B/s 
        R_C_M=1.25e9,     # 10 Gbps -> 1.25e9 B/s
        mu=2e7, 
        eta=2e5,
        B_pu_matrix=np.full((P, max_U), 40e6, dtype=float),    # 320 Mbps -> 40e6 B/s
        gamma_pu_matrix=np.full((P, max_U), 5.0, dtype=float), # SNRs
        rhoT_p=[0.2], 
        lambda_p=[0.05],
        du_fixed_delay=0.001,    # 1 ms
        mec_fixed_delay=0.005,   # 5 ms
        cloud_fixed_delay=0.05   # 50 ms
    )

    env = EnvWrapper(
        n=n,
        n_layers=n_layers,
        users_env=users_env, 
        du_caches=du_caches,
        mec_cache=mec_cache,
        latency_model=lat_model,
        lam=lam,
        theta=theta
    )

    print(
        f"Experiment ===================\n"
        f"Total users: {n_users}\n"
        f"Cache capacity: {max_capacity}MB\n"
        f"Video matrix: {n_videos}x{n_layers}x{n*n}\n"
        f"==============================\n"
    )

    _, info = env.reset()
    viewport_tiles = info['viewport_tiles']
    requested_videos = info['requested_videos']

    done = False
    results = []
    total_reward = 0.0

    for step in count():
        actions = [
            {
                'user': user,
                'video': requested_videos[user],
                'tiles': getTiles(step % n_gops, user, viewport_tiles, n),
                'gop': step % n_gops
            } for user in range(n_users)
        ]

        obs, reward, done, info = env.step(actions)

        total_reward += float(reward)

        results.append({
            "step": step,
            "total_reward": total_reward,
            "cache_hits": info["base_layer_cache_hits"] + info["enh_layer_cache_hits"],
            "cache_misses": info["base_layer_cache_misses"] + info["enh_layer_cache_misses"],
            "info": info
        })

        print(
            f"Step {step} - "
            f"Reward: {reward:.4f} - "
            f"Total Reward: {total_reward:.4f} - "
            f"Cache Hits: {info['base_layer_cache_hits'] + info['enh_layer_cache_hits']} - "
            f"Cache Misses: {info['base_layer_cache_misses'] + info['enh_layer_cache_misses']}\n"
            f"Cache Utilization: {info['cache_utilization']:.2%} - "
            f"Items in Cache: {info['cache_num_items']} - "
            f"Final Capacity: {info['cache_current_capacity']/1e6:.2f}/{info['cache_total_capacity']/1e6:.1f} MB"
        )
        
        if done: 
            break